In [ ]:
def iqft(n):
    qc = QuantumCircuit(n)
    for target in reversed(range(n)):
        for control in reversed(range(target +1, n)):
            k = control - target + 1
            angle = -2 * np.pi / (2 ** k)
            qc.cp(angle, control, target)
        qc.h(target)
    for i in range(n // 2):
        qc.swap(i, n - i - 1)
    return qc

In [ ]:
def c_amod15(a, power):
    if a not in [2, 7, 8, 11, 13]:
        raise ValueError("Pick k from the list: 2, 7, 8, 11, 13")
    qc = QuantumCircuit(4)        
    for _ in range(power):
        if a in [2, 13]:
            qc.swap(0, 1); qc.swap(1, 2); qc.swap(2, 3)
        if a in [7, 8]:
            qc.swap(2, 3); qc.swap(1, 2); qc.swap(0, 1)
        if a == 11:
            qc.swap(0, 2); qc.swap(1, 3)
        if a in [7, 11, 13]:
            for q in range(4): qc.x(q)
    return qc.to_gate(label=f"{a}^{power} mod 15")

In [ ]:
def build_shor_circuit(a, n):
    qc = QuantumCircuit(n + 4, n)
    for q in range(n):
        qc.h(q)
    qc.x(n)
    qc.barrier()

    for q in range(n):
        u_gate = c_amod15(a, 2**q)
        u_gate_control = u_gate.control()
        qc.append(u_gate_control, [q] + list(range(n, n + 4))

    qc.barrier()
    iqft_gate = iqft(n)    
    qc.compose(iqft_gate, range(n), inplace=True)

    qc.measure(range(n), range(n))
    return qc

In [ ]:
n = 8
a = 7

shor_circuit = build_shor_circuit(a, n)
sim = AerSimulator()
transpiled_circuit = transpile(shor_circuit, sim)
result = sim.run(transpiled_circuit, shots=1024).result()
print(f"Measurement results: {result.get_counts()}")